In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
data_path = "../data/feedipedia_raw_data.json"

with open(data_path, "r", encoding="utf-8") as file:
    raw_residues = json.load(file)

print(f"Number of residues: {len(raw_residues)}")

Number of residues: 78


In [7]:
from metabolic_rules import NUTRIENT_TARGETS

def get_nutrient_value(data_dict, category, nutrient):
    """Safely extracts the 'Avg' value for a given nutrient from the JSON structure."""

    try:
        value = data_dict.get(category, {}).get(nutrient, {}).get('Avg', 0)
        return float(value)
    except (KeyError, ValueError, TypeError):
        return 0.0

processed_residues = []

for item in raw_residues: 
    name = item.get('Residue_Name', 'Unknown')
    category = item.get('Main_Category', 'Unknown')
    data = item.get('Data', {})
    
    profile = {
        'Residue': name,
        'Category': category
    }
    
    for nutrient in NUTRIENT_TARGETS:
        profile[nutrient] = get_nutrient_value(data, 'Main analysis', nutrient)
        
    processed_residues.append(profile)

df_residues = pd.DataFrame(processed_residues)

df_residues.replace(0.0, np.nan, inplace=True)
df_residues = df_residues.fillna(0)

df_residues.head()

,Residue,Category,Lactose,Starch (polarimetry),Starch (enzymatic),Crude fibre,NDF,Neutral detergent fibre,ADF,Acid detergent fibre,Total sugars,Fructose
0,Barley distillers grains (ethanol),Cereal grains and by-products,0.0,0.9,0.0,13.8,60.1,0.0,23.8,0.0,0.0,0.0
1,"Brewers grains, dehydrated",Cereal grains and by-products,0.0,7.8,0.0,15.8,56.3,0.0,21.9,0.0,0.0,0.0
2,Maize bran,Cereal grains and by-products,0.0,35.0,0.0,12.3,44.2,0.0,14.5,0.0,2.8,0.0
3,Hominy feed,Cereal grains and by-products,0.0,40.5,0.0,6.5,30.7,0.0,8.9,0.0,4.8,0.0
4,Maize cobs,Cereal grains and by-products,0.0,10.7,0.0,34.9,87.8,0.0,46.3,0.0,0.0,0.0


In [8]:
features_to_scale = NUTRIENT_TARGETS

df_residues[features_to_scale] = df_residues[features_to_scale].fillna(0.0)

# Min-Max scaling (it will scale each feature to the range [0, 1])
df_scaled = df_residues.copy()

for feature in features_to_scale:
    if feature in df_scaled.columns:
        max_val = df_scaled[feature].max()
        min_val = df_scaled[feature].min()
        
        # Avoid division by zero in case max_val equals min_val
        if max_val > min_val:
            df_scaled[feature] = (df_scaled[feature] - min_val) / (max_val - min_val)
        else:
            df_scaled[feature] = 0.0


df_scaled.head()

,Residue,Category,Lactose,Starch (polarimetry),Starch (enzymatic),Crude fibre,NDF,Neutral detergent fibre,ADF,Acid detergent fibre,Total sugars,Fructose
0,Barley distillers grains (ethanol),Cereal grains and by-products,0.0,0.016854,0.0,0.191667,0.683732,0.0,0.330556,0.0,0.000000,0.0
1,"Brewers grains, dehydrated",Cereal grains and by-products,0.0,0.146067,0.0,0.219444,0.640501,0.0,0.304167,0.0,0.000000,0.0
2,Maize bran,Cereal grains and by-products,0.0,0.655431,0.0,0.170833,0.502844,0.0,0.201389,0.0,0.043682,0.0
3,Hominy feed,Cereal grains and by-products,0.0,0.758427,0.0,0.090278,0.349261,0.0,0.123611,0.0,0.074883,0.0
4,Maize cobs,Cereal grains and by-products,0.0,0.200375,0.0,0.484722,0.998862,0.0,0.643056,0.0,0.000000,0.0
